In [3]:
import os
import mne
# import PyQt6
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

# from mne_features.univariate import compute_higuchi_fd

# %matplotlib qt

In [4]:
folders = [x for x in os.listdir('OCD/') if os.path.isdir('OCD/' + x) if 'stress' not in x]
fldr2label = {folders[i]: i for i in range(len(folders))}
fldr2label

{'anxiety_ADD (47)': 0,
 'anxiety_AFD (18)': 1,
 'anxiety_general_AD (16)': 2,
 'bipolar_BPD_1(26)': 3,
 'bipolar_BPD_2(25)': 4,
 'controls (157)': 5,
 'Cyclothymia(10)': 6,
 'depression_mild (29)': 7,
 'depression_moderade (47)': 8,
 'depression_severe (32)': 9,
 'personality_disorder (56)': 10}

In [5]:
label2class = {
    0: 0, 1: 0, 2: 0, # anxiety
    3: 1, 4: 1, # bipolar
    5: 2, # control
    6: 3, # cyclothymia
    7: 4, 8: 4, 9: 4, # depression
    10: 5, # personality disorder
}

In [6]:
og_files = []
zg_files = []
og_labels = []
zg_labels = []
for fldr in folders:
    pth =  'OCD/' + fldr
    og_pths = [x for x in os.listdir(pth) if x.lower().endswith('.edf') and ('og.' in x.lower() or 'ог.' in x.lower() or 'eo.' in x.lower() or '_eo' in x.lower())]
    zg_pths = [x for x in os.listdir(pth) if x.lower().endswith('.edf') and ('zg.' in x.lower() or 'зг.' in x.lower() or 'ec.' in x.lower() or 'fon.' in x.lower() or '_ec' in x.lower() or 'eс.' in x.lower())]
    left = [x for x in os.listdir(pth) if x not in og_pths and x not in zg_pths]
    if len(left) > 0:
        print(pth, left)
    for f in og_pths:
        og_files.append(pth + '/' + f)
        og_labels.append(fldr2label[fldr])
    for f in zg_pths:
        zg_files.append(pth + '/' + f)
        zg_labels.append(fldr2label[fldr])

print(f'Number of files for open eyes: {len(og_files)}')
print(f'Number of files for closed eyes: {len(zg_files)}')

# ensured that og_files[i] is the pair for zg_files[i]

Number of files for open eyes: 461
Number of files for closed eyes: 461


In [24]:
Counter([label2class[x] for x in zg_labels])

Counter({2: 157, 4: 108, 0: 81, 5: 54, 1: 51, 3: 10})

# Calculate features

In [7]:
def compute_features(data, s_freq=250, pairs=[], n_channels=19):
    freq_bands = np.array([1, 4, 8, 13, 30])

    psd, freqs = mne.time_frequency.psd_array_welch(data, sfreq=s_freq, fmin=1, fmax=30, n_fft=512, n_per_seg=s_freq, window='hann', average=None, verbose=False)

    # compute ES and DE
    differential_entropy = np.zeros((psd.shape[0], len(freq_bands) - 1, psd.shape[2]))
    energy_spectrum = np.zeros((psd.shape[0], len(freq_bands) - 1, psd.shape[2]))

    for i in range(n_channels):
        for j in range(len(freq_bands) - 1):
            band_indices = np.where((freqs >= freq_bands[j]) & (freqs < freq_bands[j+1]))[0]
            if len(band_indices) > 0:
                band_psd = psd[i, band_indices]
                band_variance = np.mean(band_psd, axis=0)
                differential_entropy[i, j] = 0.5 * np.log(2 * np.pi * np.e * band_variance)
                energy_spectrum[i, j] = band_variance
    # compute asymmetry features
    left = differential_entropy[[x[0] for x in pairs]]
    right = differential_entropy[[x[1] for x in pairs]]
    dasm = left - right
    rasm = left / right
    return energy_spectrum, differential_entropy, dasm, rasm

In [16]:
# calculate features for the first 14s of each recording
n_channels = 19

cleaned_labels_zg = []
es_features_zg = []
de_features_zg = []
dasm_features_zg = []
rasm_features_zg = []

name_pairs = [('Fp1', 'Fp2'), ('F3', 'F4'), ('F7', 'F8'), ('C3', 'C4'), ('T3', 'T4'), ('P3', 'P4'), ('T5', 'T6'), ('O1', 'O2')]
idx_pairs = [(0, 1), (2, 4), (5, 6), (9, 11), (7, 8), (14, 16), (12, 13), (17, 18)]
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']

to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in range(len(zg_files)):
    try:
        path = zg_files[i]
        if any([x in path for x in to_skip]):
            continue
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
        # skip faulty data for now
        if 'chan' in sample.ch_names[0].lower():
            continue

        sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
        channels = sample.ch_names
        to_drop = channels[19:]

        new_idx = []
        skip = False
        for ch in channels2use:
            found = False
            for k in range(19):
                if ch in channels[k]:
                    new_idx.append(k)
                    found = True
                    break
            if not found:
                skip = True
                break
        if skip:
            continue

        s_freq = int(sample.info['sfreq'])
        data = sample.get_data()[new_idx, :int(13 * s_freq)]

        es, de, dasm, rasm = compute_features(data, s_freq, idx_pairs)
        es_features_zg.append(es)
        de_features_zg.append(de)
        dasm_features_zg.append(dasm)
        rasm_features_zg.append(rasm)
        cleaned_labels_zg.append(zg_labels[i])
    except Exception as e:
        print(e)
        print(path)

In [17]:
# calculate features for the first 14s of each recording
n_channels = 19

cleaned_labels_og = []
es_features_og = []
de_features_og = []
dasm_features_og = []
rasm_features_og = []

name_pairs = [('Fp1', 'Fp2'), ('F3', 'F4'), ('F7', 'F8'), ('C3', 'C4'), ('T3', 'T4'), ('P3', 'P4'), ('T5', 'T6'), ('O1', 'O2')]
idx_pairs = [(0, 1), (2, 4), (5, 6), (9, 11), (7, 8), (14, 16), (12, 13), (17, 18)]
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']

to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in range(len(og_files)):
    try:
        path = og_files[i]
        if any([x in path for x in to_skip]):
            continue
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
        # skip faulty data for now
        if 'chan' in sample.ch_names[0].lower():
            continue

        sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
        channels = sample.ch_names
        to_drop = channels[19:]
        sample.drop_channels(to_drop)

        new_idx = []
        skip = False
        for ch in channels2use:
            found = False
            for k in range(19):
                if ch in channels[k]:
                    new_idx.append(k)
                    found = True
                    break
            if not found:
                skip = True
                break
        if skip:
            print(f'skipped {path}')
            continue

        s_freq = int(sample.info['sfreq'])
        data = sample.get_data()[new_idx, :int(13 * s_freq)]

        es, de, dasm, rasm = compute_features(data, s_freq, idx_pairs)
        es_features_og.append(es)
        de_features_og.append(de)
        dasm_features_og.append(dasm)
        rasm_features_og.append(rasm)
        cleaned_labels_og.append(og_labels[i])
    except Exception as e:
        print(e)
        print(path)

In [18]:
es_features_og = np.array(es_features_og)
de_features_og = np.array(de_features_og)
dasm_features_og = np.array(dasm_features_og)
rasm_features_og = np.array(rasm_features_og)
es_features_zg = np.array(es_features_zg)
de_features_zg = np.array(de_features_zg)
dasm_features_zg = np.array(dasm_features_zg)
rasm_features_zg = np.array(rasm_features_zg)

cleaned_labels_og = np.array(cleaned_labels_og)
cleaned_labels_zg = np.array(cleaned_labels_zg)

# Closed eyes

In [20]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

f1_scores_macro = []
f1_scores_micro = []

for i, (train, test) in enumerate(kf.split(list(range(len(cleaned_labels_zg))), cleaned_labels_zg)):
    X_train = np.concatenate((de_features_zg[train].reshape(-1, 19 * 4 * 13), dasm_features_zg[train].reshape(-1, 8 * 4 * 13), rasm_features_zg[train].reshape(-1, 8 * 4 * 13)), axis=1)
    X_test = np.concatenate((de_features_zg[test].reshape(-1, 19 * 4 * 13), dasm_features_zg[test].reshape(-1, 8 * 4 * 13), rasm_features_zg[test].reshape(-1, 8 * 4 * 13)), axis=1)
    y_train = cleaned_labels_zg[train]
    y_test = cleaned_labels_zg[test]
    clf = SVC(kernel='linear', class_weight='balanced', random_state=92)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    f1_scores_macro.append(f1_score(y_test, preds, average='macro'))
    f1_scores_micro.append(f1_score(y_test, preds, average='micro'))

# print(f1_scores_macro)
print(f'avg macro-f1: {np.mean(f1_scores_macro)}')
# print(f1_scores_micro)
print(f'avg micro-f1: {np.mean(f1_scores_micro)}')

avg macro-f1: 0.27727772824609265
avg micro-f1: 0.4175824175824176


# Open eyes

In [21]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

f1_scores_macro = []
f1_scores_micro = []

for i, (train, test) in enumerate(kf.split(list(range(len(cleaned_labels_og))), cleaned_labels_og)):
    X_train = np.concatenate((de_features_og[train].reshape(-1, 19 * 4 * 13), dasm_features_og[train].reshape(-1, 8 * 4 * 13), rasm_features_og[train].reshape(-1, 8 * 4 * 13)), axis=1)
    X_test = np.concatenate((de_features_og[test].reshape(-1, 19 * 4 * 13), dasm_features_og[test].reshape(-1, 8 * 4 * 13), rasm_features_og[test].reshape(-1, 8 * 4 * 13)), axis=1)
    y_train = cleaned_labels_og[train]
    y_test = cleaned_labels_og[test]
    clf = SVC(kernel='linear', class_weight='balanced', random_state=92)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    f1_scores_macro.append(f1_score(y_test, preds, average='macro'))
    f1_scores_micro.append(f1_score(y_test, preds, average='micro'))
    # print(confusion_matrix(y_test, preds))
    # print()

# print(f1_scores_macro)
print(f'avg macro-f1: {np.mean(f1_scores_macro)}')
# print(f1_scores_micro)
print(f'avg micro-f1: {np.mean(f1_scores_micro)}')

avg macro-f1: 0.25962553337370253
avg micro-f1: 0.38461538461538464
